In [12]:
import pandas as pd
import re

df_raw = pd.read_excel('ordersbase.xlsx', sheet_name='Sheet2', header=None)
print(f'Рядків у файлі: {len(df_raw)}')

Рядків у файлі: 43207


In [13]:
records = []
current_order = {}

for i in range(1, len(df_raw)):
    row = df_raw.iloc[i]
    col0 = str(row[0]) if pd.notna(row[0]) else ''
    col1 = str(row[1]) if pd.notna(row[1]) else ''

    # Заголовок замовлення
    if col0.startswith('Заказ '):
        current_order = {
            'store':         str(row[2]) if pd.notna(row[2]) else '',
            'order_date':    str(row[3]) if pd.notna(row[3]) else '',
            'exchange_rate': str(row[9]) if pd.notna(row[9]) else '',
            'order_id':      str(row[12]) if pd.notna(row[12]) else '',
        }

    # Рядок товару
    is_phone = bool(re.match(r'^\+?3?8?0\d', col1.replace(' ', ''))) or \
               bool(re.match(r'^3\.8\d+e\+11', col1))

    if is_phone and col0 and not col0.startswith('Заказ'):
        phone = col1
        if re.match(r'^3\.8\d+e\+11', col1):
            try:
                phone = '+' + str(int(float(col1)))
            except:
                pass

        records.append({**current_order,
            'customer_name': col0,
            'product_name':  str(row[2]) if pd.notna(row[2]) else '',
            'currency':      str(row[8]) if pd.notna(row[8]) else '',
            'store_price':   str(row[9]) if pd.notna(row[9]) else '',
            'discount':      str(row[10]) if pd.notna(row[10]) else '',
            'charged_price': str(row[12]) if pd.notna(row[12]) else '',
            'quantity':      str(row[7]) if pd.notna(row[7]) else '',
        })

df = pd.DataFrame(records)
print(f'Товарів витягнуто: {len(df)}')
print(df.head(3))

Товарів витягнуто: 7175
          store  order_date exchange_rate    order_id       customer_name  \
0  Michael Kors  08.03.2026       44.0500  Z015863557      Ирина Титечко    
1  Michael Kors  08.03.2026       44.0500  Z015863557      Ирина Титечко    
2   Other (EUR)  05.03.2026       51.2800   EU2613995  Христя Дзержинська   

                              product_name currency store_price discount  \
0    Hudson Mini Signature Logo Flight Bag      USD      149.63       0%   
1  Hudson Signature Logo Travel Sling Pack      USD       89.25       0%   
2                                     ENDO      EUR       26.45       0%   

  charged_price quantity  
0        172.08        1  
1        102.64        1  
2         30.42        1  


In [14]:
# Правильний порядок колонок
orders = df[[
    'order_date', 'store', 'order_id', 'customer_name', 
    'product_name', 'quantity', 'currency', 'store_price', 
    'discount', 'charged_price', 'exchange_rate'
]].copy()

# Прибираємо nan
orders = orders.replace('nan', '')

# Зберігаємо
orders.to_csv('orders.csv', index=False)
print(f'Рядків: {len(orders)}')
print(orders.head(3))

Рядків: 7175
   order_date         store    order_id       customer_name  \
0  08.03.2026  Michael Kors  Z015863557      Ирина Титечко    
1  08.03.2026  Michael Kors  Z015863557      Ирина Титечко    
2  05.03.2026   Other (EUR)   EU2613995  Христя Дзержинська   

                              product_name quantity currency store_price  \
0    Hudson Mini Signature Logo Flight Bag        1      USD      149.63   
1  Hudson Signature Logo Travel Sling Pack        1      USD       89.25   
2                                     ENDO        1      EUR       26.45   

  discount charged_price exchange_rate  
0       0%        172.08       44.0500  
1       0%        102.64       44.0500  
2       0%         30.42       51.2800  


In [4]:
reg_raw = pd.read_excel('regdate.xlsx')
print(reg_raw.columns.tolist())
print(reg_raw.head(3))

['Покупатель', 'Телефон', 'email', 'Баланс (UAH)', 'Регистрация', 'Заказы', 'Сумма', 'Дата последнего заказа']
         Покупатель       Телефон                    email  Баланс (UAH)  \
0     Общий баланс: -2.791860e+03             Покупателей:        354.00   
1         Alla Alla  3.805038e+11                      NaN          0.00   
2  Юлия Пономаренко  3.805014e+11  yulya.ponka16@gmail.com          0.78   

  Регистрация  Заказы      Сумма Дата последнего заказа  
0         NaN     NaN        NaN                    NaN  
1  11.03.2020     1.0    2051.79             11.03.2020  
2  15.02.2020   104.0  197636.43             29.12.2025  


In [15]:
customers = reg_raw[['Покупатель', 'Регистрация', 'Дата последнего заказа']].copy()
customers.columns = ['customer_name', 'registration_date', 'last_order_date']

# Прибираємо перший рядок (загальний баланс)
customers = customers.dropna(subset=['registration_date'])
customers = customers.reset_index(drop=True)

customers.to_csv('customers.csv', index=False)
print(f'Клієнтів: {len(customers)}')
print(customers.head(3))

Клієнтів: 354
      customer_name registration_date last_order_date
0         Alla Alla        11.03.2020      11.03.2020
1  Юлия Пономаренко        15.02.2020      29.12.2025
2   Валерия Валерия        26.02.2020      06.01.2023


In [16]:
orders['product_name'] = orders['product_name'].str[:100]
orders.to_csv('orders.csv', index=False)
print('Готово!')

Готово!


In [18]:
print(df.columns.tolist())

['store', 'order_date', 'exchange_rate', 'order_id', 'customer_name', 'product_name', 'currency', 'store_price', 'discount', 'charged_price', 'quantity']


In [19]:
orders['total_uah'] = (df['charged_price'].astype(float) * df['exchange_rate'].astype(float)).round(2)
orders.to_csv('orders.csv', index=False)
print(orders[['charged_price', 'exchange_rate', 'total_uah']].head(5))

  charged_price exchange_rate  total_uah
0        172.08       44.0500    7580.12
1        102.64       44.0500    4521.29
2         30.42       51.2800    1559.94
3         30.42       51.2800    1559.94
4         63.48       43.6700    2772.17


In [20]:
orders = orders.rename(columns={'total_uah': 'charged_price_uah'})
orders.to_csv('orders.csv', index=False)
print(orders.columns.tolist())

['order_date', 'store', 'order_id', 'customer_name', 'product_name', 'quantity', 'currency', 'store_price', 'discount', 'charged_price', 'exchange_rate', 'charged_price_uah']


In [21]:
print(orders.columns.tolist())
print(orders['charged_price_uah'].head(3))

['order_date', 'store', 'order_id', 'customer_name', 'product_name', 'quantity', 'currency', 'store_price', 'discount', 'charged_price', 'exchange_rate', 'charged_price_uah']
0    7580.12
1    4521.29
2    1559.94
Name: charged_price_uah, dtype: float64


In [22]:
# Конвертуємо дати
orders['order_date'] = pd.to_datetime(orders['order_date'], format='%d.%m.%Y')

print(orders.dtypes)
print(orders['order_date'].head(3))

order_date           datetime64[ns]
store                        object
order_id                     object
customer_name                object
product_name                 object
quantity                     object
currency                     object
store_price                  object
discount                     object
charged_price                object
exchange_rate                object
charged_price_uah           float64
dtype: object
0   2026-03-08
1   2026-03-08
2   2026-03-05
Name: order_date, dtype: datetime64[ns]


In [23]:
orders['store_price'] = pd.to_numeric(orders['store_price'], errors='coerce')
orders['charged_price'] = pd.to_numeric(orders['charged_price'], errors='coerce')
orders['exchange_rate'] = pd.to_numeric(orders['exchange_rate'], errors='coerce')
orders['quantity'] = pd.to_numeric(orders['quantity'], errors='coerce')

print(orders.dtypes)

order_date           datetime64[ns]
store                        object
order_id                     object
customer_name                object
product_name                 object
quantity                      int64
currency                     object
store_price                 float64
discount                     object
charged_price               float64
exchange_rate               float64
charged_price_uah           float64
dtype: object


In [24]:
orders.to_csv('orders.csv', index=False)
print('Збережено!')
print(orders.head(3))

Збережено!
  order_date         store    order_id       customer_name  \
0 2026-03-08  Michael Kors  Z015863557      Ирина Титечко    
1 2026-03-08  Michael Kors  Z015863557      Ирина Титечко    
2 2026-03-05   Other (EUR)   EU2613995  Христя Дзержинська   

                              product_name  quantity currency  store_price  \
0    Hudson Mini Signature Logo Flight Bag         1      USD       149.63   
1  Hudson Signature Logo Travel Sling Pack         1      USD        89.25   
2                                     ENDO         1      EUR        26.45   

  discount  charged_price  exchange_rate  charged_price_uah  
0       0%         172.08          44.05            7580.12  
1       0%         102.64          44.05            4521.29  
2       0%          30.42          51.28            1559.94  


In [25]:
customers['registration_date'] = pd.to_datetime(customers['registration_date'], format='%d.%m.%Y', errors='coerce')
customers['last_order_date'] = pd.to_datetime(customers['last_order_date'], format='%d.%m.%Y', errors='coerce')

customers.to_csv('customers.csv', index=False)
print(customers.dtypes)
print(customers.head(3))

customer_name                object
registration_date    datetime64[ns]
last_order_date      datetime64[ns]
dtype: object
      customer_name registration_date last_order_date
0         Alla Alla        2020-03-11      2020-03-11
1  Юлия Пономаренко        2020-02-15      2025-12-29
2   Валерия Валерия        2020-02-26      2023-01-06


In [26]:
orders['order_date'] = orders['order_date'].dt.strftime('%Y-%m-%d')
orders.to_csv('orders.csv', index=False)

customers['registration_date'] = customers['registration_date'].dt.strftime('%Y-%m-%d')
customers['last_order_date'] = customers['last_order_date'].dt.strftime('%Y-%m-%d')
customers.to_csv('customers.csv', index=False)

print('Готово!')
print(orders['order_date'].head(3))

Готово!
0    2026-03-08
1    2026-03-08
2    2026-03-05
Name: order_date, dtype: object
